### Data Reading JSON

In [0]:
df_json = spark.read.format("json").option('inferSchema', True)\
    .option('header', True)\
        .option("multiLine", False)\
            .load("/Volumes/workspace/default/my_files/drivers.json")

In [0]:
df_json.display()

### Data Reading

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/")

In [0]:
df = spark.read.format("csv").option("inferSchema", True).option("header", True).load('/Volumes/workspace/default/my_files/BigMart Sales.csv')

In [0]:
df.display()

### DDL SCHEMA

In [0]:
my_ddl_schema = """
item_identifier string,
item_weight double,
item_fat_content string,
item_visibility double,
item_type string,
item_MRP double,
outlet_identifier string,
outlet_establishment_year integer,
outlet_size string,
outlet_location_type string,
outlet_type string,
item_outlet_sales double"""

In [0]:
df = spark.read.format('csv').schema(my_ddl_schema).option('header', True). load('/Volumes/workspace/default/my_files/BigMart Sales.csv')

In [0]:
df.display()

### StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('item_identifier', StringType(), True),
    StructField('item_weight', DoubleType(), True),
    StructField('item_fat_content', StringType()),
    StructField('item_visibility', DoubleType(), True),
    StructField('item_type', StringType(), True),
    StructField('item_MRP', DoubleType(), True),
    StructField('outlet_identifier', StringType(), True),
    StructField('outlet_establishment_year', IntegerType(), True    ),
    StructField('outlet_size', StringType(), True),
    StructField('outlet_location_type', StringType(), True),
    StructField('outlet_type', StringType(), True),
    StructField('item_outlet_sales', DoubleType(), True)
])

In [0]:
df = spark.read.format('csv').schema(my_struct_schema).option('header', True).load('/Volumes/workspace/default/my_files/BigMart Sales.csv')

In [0]:
df.printSchema()

### SELECT

In [0]:
df.display()

In [0]:
df_sel = df.select(col('item_identifier'), col('item_weight'), col('item_fat_content')).display()

### ALIAS

In [0]:
df.select(col('item_identifier').alias('item_ID')).display()

In [0]:
df.display()

### FILTER/WHERE

#### Scenario - 1

In [0]:
df.filter(col('item_fat_content')=='Regular').display()

#### Scenario - 2

In [0]:
df.filter((col('item_type')=='Soft Drinks') & (col("item_weight")<10)).display()

#### Scenario - 3


In [0]:
df.filter(col('outlet_size').isNull() & col('outlet_location_type').isin(['Tier 1','Tier 2'])).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('item_weight', 'Item wt').display()

### withColumn

#### Scenario - 1

In [0]:
df = df.withColumn('flag', lit('new'))

In [0]:
df.display()

In [0]:
df.withColumn('multiply', col('item_weight')*col('item_MRP')).display()

#### Scenario - 2

In [0]:
df.withColumn('Item_fat_content', regexp_replace('Item_fat_content', 'Regular', 'Reg')).withColumn('Item_fat_content', regexp_replace('Item_fat_content', 'Low Fat', 'LF')).display()

### Type Casting


In [0]:
df = df.withColumn('Item_weight', col('Item_weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort/ OrderBy

#### Scenario - 1

In [0]:
df.sort(col('Item_weight').desc()).display()

#### Scenario - 2

In [0]:
df.sort(col('item_visibility').asc()).display()

#### Scenario - 3

In [0]:
df.sort(['Item_weight', 'item_visibility'],ascending = [1,0]).display()

### Limit

In [0]:
df.limit(10).display()

### Drop

#### Scenario - 1

In [0]:
df.drop('Item_visibility').display()

#### Scenario - 2

In [0]:
df.drop('item_visibility', 'item_type').display()

### Drop Duplicates

In [0]:
df.dropDuplicates().display()

#### Scenario - 2


In [0]:
df.dropDuplicates(subset = ['item_type']).display()

In [0]:
df.distinct().display()

### UNION and UNION BYNAME

In [0]:

data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)


In [0]:
df1.display()

In [0]:
df2.display()

### UNION

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad','1'),
        ('sid','2')]
schema1 = 'name STRING, id STRING' 
df1 = spark.createDataFrame(data1,schema1)

### UNION BY NAME

In [0]:
df1.unionByName(df2).display()

### String Function

#### Initcap()

In [0]:
df.select(upper('Item_Type').alias('UPPER-ITEM_TYPE')).display()

### Date Function

In [0]:
#### Current_Date


In [0]:
df = df.withColumn('curr_date', current_date())
df.display()

#### Date_Add

In [0]:
df = df.withColumn('week_after', date_add('curr_date', 7))
df.display()

#### Date_Sub()

In [0]:
df = df.withColumn('week_before', date_sub('curr_date', 7))
df.display()

In [0]:
df = df.withColumn('week_before', date_add('curr_date', -7))
df.display()

### DATEDIFF

In [0]:
df = df.withColumn('datediff', datediff('week_after', 'curr_date'))
df.display()

### Date_Format

In [0]:
df = df.withColumn('week_before', date_format('week_before', 'dd-MM-yyyy'))
df.display()

### Handling Nulls

#### Drop Null

In [0]:
 df.dropna('all').display()

In [0]:
df.dropna('any').display()


In [0]:
df.dropna(subset='outlet_size').display()

In [0]:
df.display()

#### Filling Nulls

In [0]:
df.fillna('Not Available').display()

In [0]:
df.fillna('Not Available', subset=['outlet_size']).display()

### SPLIT and Indexing

#### SPLIT

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).display()

#### Indexing

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).display()

### EXPLODE

In [0]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type', explode('Outlet_Type')).display()

### ARRAY_CONTAINS

In [0]:
df_exp.display()

In [0]:
df_exp.withColumn('Type1_flag', array_contains('Outlet_Type', 'Type1')).display()

### GROUP BY

#### Scenario 1

In [0]:
df.display()

In [0]:
df.groupBy('item_type').agg(sum('item_MRP')).display()

#### Scenario 2

In [0]:
df.groupBy('item_type').agg(avg('item_MRP')).display()

#### Scenario 3

In [0]:
df.groupBy('item_type', 'outlet_size').agg(sum('item_MRP').alias('Total_MRP')).display()

#### Scenario 4

In [0]:
df.groupBy('item_type', 'outlet_size').agg(sum('item_MRP'), avg('item_MRP')).display()

### COLLECT LIST

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

### PIVOT

In [0]:
df.groupBy('item_type').pivot('outlet_size').agg(avg('item_MRP')).display()

### WHEN OTHERWISE

#### Scenario 1

In [0]:
df = df.withColumn('veg_flag', when(col('item_type')=='Meat', 'Non-veg').otherwise('Veg'))
df.display()

#### Scenario 2

In [0]:
df.withColumn('vegflag', when((col('veg_flag')=='Veg') & (col("item_MRP")< 100), 'Veg inexpensive').when((col('veg_flag')=='Veg') & (col("item_MRP")>= 100), 'Veg expensive').otherwise('Non veg')).display()

### Joins

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

#### Inner Join

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'inner').display()

###

#### Left Join

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'left').display()

#### Right Join

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'right').display()

#### Anti Join

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'anti').display()

### WINDOW FUNCTION

### ROW NUMBER

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('item_identifier'))).display()

### RANK VS DENSE RANK

In [0]:
df.withColumn('rankCol', rank().over(Window.orderBy(col('item_identifier').desc()))).withColumn('denseRankCol', dense_rank().over(Window.orderBy(col('item_identifier').desc()))).display()

### CUMMULATIVE SUM

In [0]:
df.withColumn('cumsum', sum('item_MRP').over(Window.orderBy('item_type'))).display()

In [0]:
df.withColumn('cumsum', sum('item_MRP').over(Window.orderBy('item_type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('cumsum', sum('item_MRP').over(Window.orderBy('item_type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

### USER DEFINED FUNCTIONS

#### STEP 1

In [0]:
def my_func(x):
  return x*x

#### STEP 2

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol', my_udf('Item_MRP')).display()

### DATA WRITING

#### CSV

In [0]:
df.write.format('csv').save('/Volumes/workspace/default/my_files/data.csv')

#### DATA WRITING MODES

#### APPEND

In [0]:
df.write.format('csv').mode('append').option('path', '/Volumes/workspace/default/my_files/data.csv').save()

#### OVERWRITE

In [0]:
df.write.format('csv').mode('overwrite').option('path', '/Volumes/workspace/default/my_files/data.csv').save()

#### ERROR

In [0]:
df.write.format('csv').mode('error').option('path', '/Volumes/workspace/default/my_files/data.csv').save()

#### IGNORE


In [0]:
df.write.format('csv').mode('ignore').option('path', '/Volumes/workspace/default/my_files/data.csv').save()

### PARQUET

In [0]:
df.write.format('parquet').mode('overwrite').option('path', '/Volumes/workspace/default/my_files/data.parquet').save()

### TABLE

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('my_table')

In [0]:
df.display()

### SPARK SQL

#### createTempView

In [0]:
df.createTempView("myView")

In [0]:
%sql
select * from myView where item_fat_content == 'Low Fat'

In [0]:
df_sql = spark.sql("select * from myView where item_fat_content == 'Low Fat'")

In [0]:
df_sql.display()